# 5.1 Memory networks

This notebook accompanies Section 5.1 of [*Community Detection with the Map Equation and Infomap: Theory and Applications*](https://doi.org/10.1145/3779648).

In a first-order network the random walker has no memory: its next step depends only on the current physical node, never on where it came from. A walker at a physical node moves to a neighbour with probability proportional to the link weights.

That assumption hides patterns whenever flow depends on its history. Air traffic is one example. A traveller who flies into a transit airport tends to fly back out toward their origin, so the next leg depends on the previous one. A memory network captures this by making the walker's next step depend on the $k-1$ previously visited physical nodes:

$$P_{uv} = P(v \mid u, u_{-1}, \dots, u_{-k+1}),$$

where $u$ is the current node and $u_{-1}$ the previous one. The map equation handles such models by encoding the walk over *state nodes* rather than physical nodes.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

import infomap


## Model memory with state networks

A memory network represents history with *state nodes*. Each physical node holds one or more state nodes, and a state node records some aspect of the walk, such as the previously visited physical node. Links connect state nodes, so the walk runs on the state level while the modules it reveals live on the physical level.



The figure shows five physical nodes $i, j, k, l, m$. The two triangles share the central physical node $i$, which holds two state nodes (red). Each state node keeps most of the flow on its own side: the left state node sends flow back toward $j$ and $k$, the right state node back toward $l$ and $m$. A first-order network would mix the two triangles at $i$ and blur the split.

Infomap partitions the network at the state level and places the left and right state nodes in different modules. Because both belong to physical node $i$, that physical node ends up in both modules, so the two communities overlap on the physical level.


In [ ]:
network = """
# The state network above
*Vertices 5
#node_id name
1 "i" 
2 "j" 
3 "k" 
4 "l" 
5 "m" 
*States
#state_id node_id name
1 1 "α~_i" 
2 2 "β~_j" 
3 3 "γ~_k" 
4 1 "δ~_i" 
5 4 "ε~_l" 
6 5 "ζ~_m" 
*Links
#source target weight
1 2 0.8 
1 3 0.8 
1 5 0.2 
1 6 0.2 
2 1 1 
2 3 1 
3 1 1 
3 2 1 
4 5 0.8 
4 6 0.8 
4 2 0.2 
4 3 0.2 
5 4 1 
5 6 1 
6 4 1 
6 5 1 
"""
with open("data/state_network.net", "w") as fp:
    fp.write(network)

In [ ]:
im = infomap.Infomap(silent=True, seed=123)
im.read_file("data/state_network.net")
im.run()
print(f"Found {im.num_top_modules} modules with codelength {im.codelength:.8f} bits")
im.get_dataframe(["node_id", "name", "state_id", "module_id", "flow"]).set_index(
    "state_id"
)

The table assigns the two state nodes of physical node $i$ to different modules, so the partition splits into two communities that overlap on $i$.

You can also build a state network directly through the Infomap API instead of reading it from a file.

In [ ]:
im = infomap.Infomap(two_level=True, silent=True, seed=123)

# Set names on physical nodes
im.set_name(1, "PRE")
im.set_name(2, "SCIENCE")
im.set_name(3, "PRL")
im.set_name(4, "BIO")

# First number is state id, second is node id (physical node)
im.add_state_node(0, 1)
im.add_state_node(1, 2)
im.add_state_node(2, 3)
im.add_state_node(3, 2)
im.add_state_node(4, 2)
im.add_state_node(5, 4)

# Links connect state nodes
im.add_link(0, 1)
im.add_link(1, 2)
im.add_link(3, 2)
im.add_link(4, 5)

im.run()

print(f"Found {im.num_top_modules} modules with codelength {im.codelength:.8f} bits")

im.get_dataframe(["node_id", "name", "state_id", "module_id", "flow"]).set_index(
    "state_id"
)